# **Recommendation Metrics**

#### evaluate genre precision, BPM accuracy, and acoustic similarity across diverse query tracks

## **Phase 1: Setup**
#### load index, vectors, and master parquet

In [1]:
import faiss
import json
import numpy as np
import pandas as pd
import time
from pathlib import Path

# Paths
index_dir     = Path(r"D:\music_data\index")
emb_dir       = Path(r"D:\music_data\embeddings")
processed_dir = Path(r"D:\music_data\processed")
out_path      = Path(r"D:\music_data\metrics_data.json")

In [2]:
# load FAISS index
index = faiss.read_index(str(index_dir / "faiss_ivfpqV3.index"))
index.nprobe = 64

with open(index_dir / "track_id_map_V3.json") as f:
    track_ids = json.load(f)

print(f"Index: {index.ntotal} | track_ids: {len(track_ids)}")

Index: 106412 | track_ids: 106412


In [3]:
# load embeddings
npy_files = sorted(emb_dir.glob("*.npy"), key=lambda p: int(p.stem))
vectors   = np.stack([np.load(f) for f in npy_files]).astype(np.float32)
faiss.normalize_L2(vectors)
print(f"Vectors: {vectors.shape}")

Vectors: (106412, 512)


In [4]:
# load master parquet
df = pd.read_parquet(processed_dir / "ab_fma_v3.parquet")
print(df.columns.tolist())
print(f"Tracks: {len(df)}")

['track_id', 'title', 'artist', 'genre', 'duration', 'subset', 'bpm', 'energy', 'key']
Tracks: 106407


## **Phase 2: Find Diverse Query Tracks**
#### pick one representative track per genre for balanced evaluation

In [5]:
# check genre distribution
genre_counts = df['genre'].value_counts()
print("Top 20 genres in dataset:")
print(genre_counts.head(20))

Top 20 genres in dataset:
genre
Rock                   14172
Experimental           10603
Electronic              9358
Hip-Hop                 3541
Folk                    2801
Pop                     2331
Instrumental            2077
International           1375
Classical               1230
Jazz                     571
Old-Time / Historic      549
Spoken                   421
Country                  194
Soul-RnB                 175
Blues                    110
Easy Listening            24
Name: count, dtype: int64


In [6]:
# find one track per target genre that exists in the FAISS index
target_genres = ['Hip-Hop', 'Electronic', 'Rock', 'Jazz', 'Folk', 'Classical', 'Pop', 'Experimental']

print(f"{'Genre':20s}  {'track_id':>10}  title")
print("-" * 60)

for genre in target_genres:
    matches = df[df['genre'] == genre]
    if matches.empty:
        print(f"  {genre:18s}  NOT FOUND")
        continue
    for _, row in matches.iterrows():
        if row['track_id'] in track_ids:
            print(f"  {genre:18s}  {row['track_id']:>8d}  '{row['title']}'")
            break

Genre                   track_id  title
------------------------------------------------------------
  Hip-Hop                    2  'Food'
  Electronic               384  'Baja Jones'
  Rock                     135  'Father's Day'
  Jazz                     144  'Wire Up'
  Folk                     139  'CandyAss'
  Classical               4850  'Preludes, Book 2 - '
  Pop                       10  'Freeway'
  Experimental             137  'Side A'


## **Phase 3: Run Metrics**
#### genre precision, BPM proximity, acoustic similarity score, and latency
#### update QUERY_IDS below using the track_ids from Phase 2

In [13]:
# ── update these with diverse track IDs from Phase 2 ──────────
QUERY_IDS = [2, 384, 135 , 139, 135, 10,4850,137]   # replace with diverse genre IDs
FAISS_K   = 50
FINAL_K   = 10

In [14]:
# helpers
def get_parquet_features(fma_id):
    row = df[df['track_id'] == fma_id]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        'bpm':    float(r.get('bpm',    0) or 0),
        'energy': float(r.get('energy', 0) or 0),
        'key':    float(r.get('key',    0) or 0),
        'genre':  r.get('genre', ''),
    }

def librosa_score(q, c):
    if not q or not c:
        return 0.0
    bpm_sim    = 1 - min(abs(q['bpm']    - c['bpm'])    / 200, 1)
    energy_sim = 1 - min(abs(q['energy'] - c['energy']) / 1,   1)
    key_sim    = 1.0 if q['key'] == c['key'] else 0.5
    return round(bpm_sim * 0.4 + energy_sim * 0.4 + key_sim * 0.2, 4)

In [15]:
# run recommendation + collect metrics
results = []

for fma_id in QUERY_IDS:
    if fma_id not in track_ids:
        print(f"skipping {fma_id} — not in index")
        continue

    pos       = track_ids.index(fma_id)
    query_vec = vectors[pos].reshape(1, -1)
    q_feats   = get_parquet_features(fma_id)
    q_label   = df[df['track_id'] == fma_id]['title'].values
    q_label   = str(q_label[0]) if len(q_label) else str(fma_id)

    t0 = time.perf_counter()
    distances, indices_out = index.search(query_vec, FAISS_K + 1)
    t1 = time.perf_counter()

    faiss_pairs = [
        (track_ids[idx], float(dist))
        for idx, dist in zip(indices_out[0], distances[0])
        if track_ids[idx] != fma_id
    ][:FAISS_K]

    scored = [
        (tid, fscore, librosa_score(q_feats, get_parquet_features(tid)))
        for tid, fscore in faiss_pairs
    ]
    scored.sort(key=lambda x: x[2], reverse=True)
    t2 = time.perf_counter()

    candidates = []
    for rank_l, (tid, fscore, lscore) in enumerate(scored[:FINAL_K]):
        rank_f = next(i for i, (t, s) in enumerate(faiss_pairs) if t == tid)
        candidates.append({
            "track_id":      tid,
            "rank_faiss":    rank_f + 1,
            "rank_librosa":  rank_l + 1,
            "faiss_score":   round(fscore, 4),
            "librosa_score": lscore,
        })

    results.append({
        "query_track_id":    fma_id,
        "query_label":       q_label,
        "query_genre":       q_feats['genre'] if q_feats else 'unknown',
        "total_latency_ms":  round((t2 - t0) * 1000, 1),
        "faiss_latency_ms":  round((t1 - t0) * 1000, 1),
        "rerank_latency_ms": round((t2 - t1) * 1000, 1),
        "candidates":        candidates,
    })

    print(f"✓ {q_label:30s}  [{q_feats['genre']}]  "
          f"FAISS: {round((t1-t0)*1000,1):6.1f}ms  "
          f"rerank: {round((t2-t1)*1000,1):6.1f}ms  "
          f"total: {round((t2-t0)*1000,1):6.1f}ms")

with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved → {out_path}")

✓ Food                            [Hip-Hop]  FAISS:    1.6ms  rerank:   42.2ms  total:   43.8ms
✓ Baja Jones                      [Electronic]  FAISS:    0.8ms  rerank:   33.4ms  total:   34.2ms
✓ Father's Day                    [Rock]  FAISS:    0.7ms  rerank:   31.5ms  total:   32.3ms
✓ CandyAss                        [Folk]  FAISS:    0.8ms  rerank:   30.6ms  total:   31.4ms
✓ Father's Day                    [Rock]  FAISS:    0.7ms  rerank:   31.4ms  total:   32.1ms
✓ Freeway                         [Pop]  FAISS:    0.8ms  rerank:   28.1ms  total:   28.9ms
✓ Preludes, Book 2 -              [Classical]  FAISS:    0.7ms  rerank:   31.2ms  total:   31.9ms
✓ Side A                          [Experimental]  FAISS:    0.8ms  rerank:   33.4ms  total:   34.1ms

Saved → D:\music_data\metrics_data.json


## **Phase 4: Compute Accuracy Metrics**
#### genre precision, avg BPM diff, avg acoustic similarity score

In [16]:
# compute metrics per query
rows = []

for q in results:
    q_row       = df[df['track_id'] == q['query_track_id']].iloc[0]
    query_genre = q_row['genre']
    query_bpm   = float(q_row['bpm'] or 0)

    genre_matches, bpm_diffs, lscores = [], [], []

    for c in q['candidates']:
        c_row = df[df['track_id'] == c['track_id']]
        if c_row.empty:
            continue
        c_row = c_row.iloc[0]
        genre_matches.append(int(c_row['genre'] == query_genre))
        bpm_diffs.append(abs(float(c_row['bpm'] or 0) - query_bpm))
        lscores.append(c['librosa_score'])

    rows.append({
        'query':             q['query_label'],
        'query_genre':       query_genre,
        'genre_precision':   round(np.mean(genre_matches) * 100, 1),
        'avg_bpm_diff':      round(np.mean(bpm_diffs), 1),
        'avg_librosa_score': round(np.mean(lscores), 4),
        'total_latency_ms':  q['total_latency_ms'],
    })

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))

print("\n── Overall ──────────────────────────────────────")
print(f"Genre precision:   {df_metrics['genre_precision'].mean():.1f}%")
print(f"Avg BPM diff:      {df_metrics['avg_bpm_diff'].mean():.1f}")
print(f"Avg librosa score: {df_metrics['avg_librosa_score'].mean():.4f}")
print(f"Avg latency:       {df_metrics['total_latency_ms'].mean():.1f}ms")

              query  query_genre  genre_precision  avg_bpm_diff  avg_librosa_score  total_latency_ms
               Food      Hip-Hop             70.0          14.7             0.8610              43.8
         Baja Jones   Electronic              0.0          11.3             0.8750              34.2
       Father's Day         Rock             40.0          13.1             0.8918              32.3
           CandyAss         Folk              0.0          13.7             0.9218              31.4
       Father's Day         Rock             40.0          13.1             0.8918              32.1
            Freeway          Pop              0.0           8.2             0.9059              28.9
Preludes, Book 2 -     Classical             90.0           8.0             0.8893              31.9
             Side A Experimental             10.0           4.4             0.8795              34.1

── Overall ──────────────────────────────────────
Genre precision:   31.2%
Avg BPM diff:  